<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex09.1-burgers-2d/Ex09.1_pipe_startup_new.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_09.1 — The Pump Starts

**Deep Learning for Engineering · Aalborg University · Part 2 · paired with L9.1 · Laminar Flow**

A liquid cold plate carries a serpentine of copper tube, and a pump drives a
water–glycol coolant through it. When the pump starts, the coolant in the
tube is at rest; it is pushed by the pressure, held back by the wall, and in a
few seconds settles into the velocity profile that every pipe-flow formula
assumes - Hagen–Poiseuille's parabola, from which Darcy's friction factor
$f = 64/\mathrm{Re}$ follows. How does the velocity across the tube grow, and
how long until the flow is steady? The start-up has an **exact solution**, so
finite differences and a physics-informed network are scored against the
truth. Then the same network finds the coolant's **viscosity** - the number
that is hardest to know, because it changes threefold with temperature and
with the glycol in the mix - from a flow meter's record taken before the flow
had settled.

| | what you do | TODO |
|---|---|---|
| **1 – 3** | the problem, its data and its physics: Newton's law in a tube, the parabola, Darcy, the start-up | — |
| **4** | the exact solution, and finite differences across the tube for an agreed 1e-03 | — |
| **5** | a PINN with the wall and the start built in, and the comparison | the trial function, the residual |
| **6** | the inverse problem: the coolant's viscosity from a flow meter | — |
| **7** | what the notebook says | — |
| **8 – 9** | your report, and the mini project proposal | your answers |

About a minute and a half on a CPU, most of it the network's two trainings. Numbers
are printed with at most two decimals, errors as powers of ten.

---

## 0 · Setup

In [ ]:
# files-cell v2 ----------------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'pipe.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex09.1-burgers-2d/"
ON_COLAB = "google.colab" in sys.modules
for f in FILES:
    if ON_COLAB or not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
for _m in ("course_core", "pinn_core", "pipe"):
    sys.modules.pop(_m, None)
print("files ready:", ", ".join(FILES))

In [ ]:
# --- setup ---------------------------------------------------------------
import os, time
for f in ("course_core.py", "pinn_core.py", "pipe.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"
from pinn_core import *                                  # noqa: F401,F403
import course_core as cc
import pipe as pp
import numpy as np, torch, matplotlib.pyplot as plt

cc.OUTPUT_DIR = "Ex09.1_outputs"                              # where the report is written
set_seed(88)  # fix the random start so a run repeats
print("torch", torch.__version__, " device:", DEVICE, " dtype:", torch.get_default_dtype())

def middle(fn, reps=5):                                    # the middle of five timings, after a warm-up
    fn(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

def online(fn, reps=5):                                    # the same, with the GPU made to finish before the clock stops
    def run():
        fn()
        if DEVICE.type == "cuda": torch.cuda.synchronize()
    return middle(run, reps)

---

## 1 · The problem

A cold plate under power electronics is cooled by a serpentine of copper tube
pressed into it: straight legs joined by U-bends. A pump drives 0.3 L/min of
coolant through it. Look at **one straight leg**, far enough from the bends
that the flow is the same at every cross-section. At $t = 0$ the pump starts:
a pressure drop $G$ per metre is applied along the tube, and held.

Wanted: the axial velocity $u$ at every radius $r$ of the tube, over the
first four seconds; how long before it is steady; and the pressure drop and
friction factor once it is.

## 2 · The data

| | |
|---|---|
| tube | copper, 6 mm inside diameter ($R$ = 3 mm) |
| coolant | 50/50 water–ethylene glycol at 40 °C: $\rho$ = 1065 kg/m³, $\mu$ = 2.5 mPa·s |
| flow | 0.3 L/min, a mean speed of 0.18 m/s |
| start | the coolant at rest; the pressure drop $G$ applied at $t = 0$ and held |
| window | 0 to 4 s |

**These are typical values, typed from memory, not measurements**: check
them before you quote a result. The pressure drop is taken to be held from
$t = 0$: a pump settles in a small fraction of the seconds the coolant takes
to follow it.

## 3 · The physics

**Newton's law for the liquid** (L9.1's *Navier–Stokes*): mass times
acceleration equals the pressure force plus the viscous force. In a straight
tube, far from the bends, the velocity points along the tube and changes only
across it, $u(r, t)$. Then the convective acceleration $(\mathbf{u}\cdot\nabla)\mathbf{u}$ is
zero - a parcel never moves into faster or slower liquid - and only

$$\rho\,u_t = G + \mu\Big(u_{rr} + \frac{u_r}{r}\Big), \qquad u = 0 \text{ on the wall}, \qquad u = 0 \text{ at } t = 0$$

is left: the push of the pressure, against the drag of the liquid's own
viscosity. The bracket is the Laplacian in a round tube.

**Steady**, $u_t = 0$, and the solution is **Hagen–Poiseuille's parabola**,

$$u = \frac{G R^2}{4\mu}\Big(1 - \frac{r^2}{R^2}\Big), \qquad \bar u = \frac{G R^2}{8\mu}$$

with the mean $\bar u$ half the centre speed. **Darcy's friction factor**
writes the pressure drop as $G = f\,\frac{1}{D}\,\frac{\rho \bar u^2}{2}$;
put the parabola's $G = 8\mu\bar u/R^2$ into it and $f = 64/\mathrm{Re}$, with
$\mathrm{Re} = \rho\bar u D/\mu$. That is where the formula of every laminar
pipe calculation comes from.

**The start-up** has an exact solution too (Szymański, 1932): the parabola,
less the part that has not yet arrived, as a sum of Bessel functions, each
dying away at its own rate. The slowest dies in about $R^2/\nu$ = 3.8 s, with
$\nu = \mu/\rho$ the kinematic viscosity: that is the time the pump's push
takes to reach the axis against the drag.

**Scaled units.** Velocity over the steady centre speed $V = GR^2/4\mu$, time
over the 4 s window, and the radius through $s = (r/R)^2$, which runs from 0
on the axis to 1 on the wall. In $s$ the equation reads

$$u_t = A + 4C\,\big(s\,u_{ss} + u_s\big), \qquad C = \frac{\nu \cdot 4\ \mathrm{s}}{R^2} = 1.04, \qquad A = 4C$$

$C$ is the viscosity in these units, and $A$ the pressure's push. The axis
needs no special care in $s$: every function of $s$ is symmetric about it.

Two checks before anything is solved: $\mathrm{Re} \approx 450$, far below
the 2300 at which pipe flow turns turbulent; and after each bend the parabola
needs about $0.05\,\mathrm{Re}\,D \approx$ 14 cm to re-form (the *entrance
length*), so the legs must be longer than that for this notebook to describe
them. The cell below prints the data and draws the exact start-up.

In [ ]:
pp.describe_problem()
s = np.linspace(0, 1, 201); r_mm = 1e3 * pp.R_TUBE * np.sqrt(s)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for t_s in (0.1, 0.3, 0.6, 1.0, 2.0, 4.0):
    u = pp.V * pp.exact(s, t_s / pp.T_END)                     # the exact velocity, m/s
    ax[0].plot(np.r_[-r_mm[::-1], r_mm], np.r_[u[::-1], u], label=f"{t_s:g} s")
ax[0].plot(np.r_[-r_mm[::-1], r_mm], pp.V * np.r_[1 - s[::-1], 1 - s], "k--", lw=1, label="the parabola")
ax[0].set_xlabel("r  [mm]"); ax[0].set_ylabel("u  [m/s]"); ax[0].legend(frameon=False, fontsize=8, ncol=2)
t = np.linspace(0, 1, 201)
ax[1].plot(t * pp.T_END, pp.V * pp.exact_mean(t), "k", lw=2)
ax[1].axhline(pp.U_MEAN, color="0.5", ls="--", lw=1, label="steady mean")
ax[1].set_xlabel("t  [s]"); ax[1].set_ylabel("mean speed  [m/s]"); ax[1].legend(frameon=False)
plt.tight_layout(); plt.show()

**What you should see.** Re = 452; a pressure drop of 393 Pa per metre;
Darcy's $f$ = 0.14; an entrance length of 136 mm; $R^2/\nu$ = 3.83 s. On the
left the profile at six instants: **flat in the middle and steep at the
wall** at first - the whole core is pushed at once, and only the liquid near
the wall feels the drag - then rounding into the parabola. On the right the
mean speed, half-way to steady after about half a second and within a per
cent of it after about three.

---

## 4 · The reference, and finite differences

**The reference** is the exact solution, `pp.exact(s, t)`: Szymański's series
summed to 400 terms, which is exact to rounding for every $t$ in the window.

**Finite differences.** Nodes from the axis to the wall, $h$ apart; at each,
the derivatives replaced by differences between neighbours,
$u_{rr} \approx (u_{j+1} - 2u_j + u_{j-1})/h^2$ and
$u_r \approx (u_{j+1} - u_{j-1})/2h$; on the axis, where $u_r/r$ becomes
$u_{rr}$, the Laplacian is $2u_{rr}$, with the node beyond the axis the
mirror of the one before it. In time, Crank–Nicolson: the average of the old
and the new right-hand side, so one sparse solve per step with the matrix
factorised once.

**The agreed accuracy is 1e-03** of the steady centre speed - 0.35 mm/s - at
every radius and every instant. The cell below refines the grid and the
step together until it is met.

In [ ]:
TS = np.linspace(0, 1, 41)                                    # the 41 instants every method is scored at, 0.1 s apart
TARGET = 1e-3
def fdm_error(n, steps):
    eta, ts, U = pp.fdm(n, steps)                             # nodes from the axis to the wall, Crank-Nicolson
    return np.abs(U - np.array([pp.exact(eta ** 2, t) for t in ts])).max()
for n, steps in ((11, 40), (21, 80), (41, 160), (81, 320)):
    err = fdm_error(n, steps); sec = online(lambda: pp.fdm(n, steps), reps=3)
    print(f"  {n:3d} nodes, {steps:3d} steps: worst error {err:.2e}, one run {1e3 * sec:.1f} ms" + ("   <- meets 1e-03" if err <= TARGET else ""))
N_FDM, STEPS = 21, 80                                         # the coarsest grid that meets it

eta = np.linspace(0, 1, N_FDM); a = np.linspace(0, 2 * np.pi, 200)
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
for e in eta[1:]:
    ax[0].plot(e * np.cos(a), e * np.sin(a), color="0.75", lw=0.6)   # each node of the grid is a ring of the tube
ax[0].plot(eta, 0 * eta, "o", ms=4, color="tab:blue"); ax[0].set_aspect("equal"); ax[0].axis("off")
ax[0].set_title(f"finite differences: {N_FDM} nodes from the axis to the wall", fontsize=9)
pts_show = np.random.default_rng(0).uniform(0, 1, (600, 2)) ** np.array([1.0, 2.0])
ax[1].plot(np.sqrt(pts_show[:, 0]), pts_show[:, 1] * pp.T_END, ".", ms=2, color="tab:orange")
ax[1].set_xlabel("r / R"); ax[1].set_ylabel("t  [s]"); ax[1].set_title("the PINN of section 5: points in (r, t)", fontsize=9)
plt.tight_layout(); plt.show()

**What you should see.** The error falling by about four with every halving -
4.76e-03, 9.08e-04, 2.68e-04, 7.15e-05 - and **21 nodes with 80 steps the
first under 1e-03**, in about a millisecond. Then the two ways of placing the
unknowns: the grid is a set of rings across the tube, marched in time; the
network of the next section is asked the equation at points scattered over
radius *and* time at once - more of them early, where the profile changes
fastest. (The points are uniform in $s$, which is uniform in the area of the
section, so they thin out towards the axis.)

---

## 5 · A physics-informed network, and the comparison

**The network** takes $(s, t)$ and returns the scaled velocity. It is four
layers of 32 tanh neurons, trained on 3000 collocation points.

**The wall and the start are built in.** The trial function

$$\hat u(s, t) = (1 - s)\; t\; \mathcal{N}(s, t)$$

is exactly zero on the wall ($s = 1$) and exactly zero at the start
($t = 0$), whatever the network does: **hard enforcement**, one factor for
each condition. Taking $s = (r/R)^2$ as the input builds in the third
condition, symmetry about the axis, for any weights. Nothing is left for a
loss term but the equation itself.

**TODO 1** is the trial function; **TODO 2** the residual. The equation is
divided by $C$ = 1.04, the scaled viscosity of section 3, so that the viscous
term is of size one: $(\hat u_t - A)/C - 4(s\,\hat u_{ss} + \hat u_s)$.

In [ ]:
class Tube(torch.nn.Module):                                  # a network: layers in __init__, the computation in forward
    def __init__(self):
        super().__init__()
        self.net = MLP(n_in=2, n_hidden=32, n_layers=4)       # dense tanh network: s and t in
    def forward(self, p):
        s, t = p[:, 0:1], p[:, 1:2]
        # TODO 1: the trial function - (1 - s) times t times the network.
        #
        #   return (1 - s) * t * self.net(p)                      # zero on the wall and at the start
        raise NotImplementedError("Write the trial function")

In [ ]:
# TODO 2: the residual of the scaled equation, divided by c.
#
#   def residual(model, p, c, a):
#       u = model(p)
#       du = grad(u, p); u_s, u_t = du[:, 0:1], du[:, 1:2]        # column 0 is s, column 1 is t
#       u_ss = grad(u_s, p)[:, 0:1]
#       s = p[:, 0:1]
#       return (u_t - a) / c - 4 * (s * u_ss + u_s)
raise NotImplementedError("Write residual(model, p, c, a)")

In [ ]:
rng = np.random.default_rng(0)
pts = to_tensor(np.c_[rng.uniform(0, 1, 3000), rng.uniform(0, 1, 3000) ** 2], requires_grad=True)   # (s, t), more early
model = Tube().to(DEVICE)                                     # four layers of 32
describe(model.net, n_collocation=len(pts))                   # the parameter count and the collocation-to-parameter ratio
loss_fn = lambda: residual(model, pts, pp.C_SCALED, pp.A_SCALED).pow(2).mean()   # the loss: the equation, nothing else
t0 = time.perf_counter()
# Adam first, then L-BFGS: the Part 2 training recipe
train_two_stage(model, loss_fn, adam_steps=500, lbfgs_steps=150, lr=2e-3, report_every=0)
train_time = time.perf_counter() - t0
print(f"trained in {train_time:.0f} s")

In [ ]:
SG, TG = np.meshgrid(np.linspace(0, 1, 101), TS)              # 101 radii x 41 instants
truth = pp.exact(SG, TG)
grid_in = to_tensor(np.c_[SG.ravel(), TG.ravel()])                # the same radii and instants, as network inputs
def pinn_field():
    with torch.no_grad():  # no gradient tracking: evaluation only
        return to_numpy(model(grid_in)).reshape(SG.shape)     # the trained network at every radius and instant
e_pin = np.abs(pinn_field() - truth).max()
eta, ts, U = pp.fdm(N_FDM, STEPS)
e_fdm = np.abs(U - np.array([pp.exact(eta ** 2, t) for t in ts])).max()
t_fdm = online(lambda: pp.fdm(N_FDM, STEPS))                  # one run of the grid, the whole window, on this runtime
t_pin = online(pinn_field)                                    # one evaluation of the trained network, the whole window, on this runtime

# the time column is what runs ONLINE: one grid run against one evaluation of the trained network, on the same runtime.
# The training is offline, paid once, and reported on its own line.
print(f"{'':34s}{'unknowns':>10s}{'worst error':>14s}{'online':>12s}")
print(f"  {'exact series, 400 terms':32s}{'-':>10s}{'-':>14s}{'-':>12s}")
print(f"  {f'finite differences, {N_FDM} x {STEPS}':32s}{N_FDM:10d}{e_fdm:14.2e}{1e3 * t_fdm:9.2f} ms")
print(f"  {'PINN, 4 x 32':32s}{parameter_count(model):10d}{e_pin:14.2e}{1e3 * t_pin:9.2f} ms")
print(f"  the network's training, offline: {train_time:.0f} s" +
      (f", repaid after {train_time / (t_fdm - t_pin):.0f} runs" if t_fdm > t_pin else ", never repaid on time alone"))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for k, c in ((4, "tab:blue"), (10, "tab:orange"), (40, "tab:green")):
    ax[0].plot(np.sqrt(SG[k]), pp.V * truth[k], color=c, lw=2, label=f"exact, {TS[k] * pp.T_END:.1f} s")
    ax[0].plot(eta, pp.V * U[k], "s", ms=3, color=c)
    ax[0].plot(np.sqrt(SG[k]), pp.V * pinn_field()[k], ":", color="k", lw=1.2)
ax[0].set_xlabel("r / R"); ax[0].set_ylabel("u  [m/s]"); ax[0].legend(frameon=False, fontsize=8)
ax[0].set_title("lines exact, squares the grid, dots the network", fontsize=9)
ax[1].semilogy(TS * pp.T_END, np.abs(U - np.array([pp.exact(eta ** 2, t) for t in ts])).max(axis=1), label="finite differences")
ax[1].semilogy(TS * pp.T_END, np.abs(pinn_field() - truth).max(axis=1), label="PINN")
ax[1].axhline(TARGET, color="k", lw=0.8, ls="--", label="agreed 1e-03")
ax[1].set_xlabel("t  [s]"); ax[1].set_ylabel("worst error across the tube"); ax[1].legend(frameon=False)
plt.tight_layout(); plt.savefig(cc.output_path("Ex09.1_report.png"), dpi=150); plt.show()

**What you should see.** Both within the agreed 1e-03 of the exact velocity
at every radius and instant: finite differences 9.08e-04 with 21 unknowns,
the network about 6e-04 with some 3 300 weights. **Online** the trained
network gives the whole window in about 0.4 ms and the grid in about 1 ms;
but the network's half-minute of training is repaid only after some
60 000 runs. In the plot the squares and the dots sit on the exact lines; the
grid's error is largest early, when the profile is steep at the wall, and the
network's is spread over the window.

**By accuracy** the two are close, the network a little ahead; **by time to a
first answer** the grid wins by far, because it needs no training. For a
straight tube with an exact solution there is no reason to prefer the
network - which is why the next section asks it a different question.

---

## 6 · The inverse problem: the coolant's viscosity

The viscosity was the one number in section 2 that is hard to know. A
water–glycol mix is three times as viscous at 20 °C as at 80 °C, and the
glycol fraction drifts as a loop is topped up with water. Suppose the loop
was filled with a stronger mix than specified, and the plate is cooler: the
true viscosity is **3.5 mPa·s**, not 2.5. What is known: the pressure drop
$G$ the pump applies, the density, and a **flow meter**'s record of the mean
speed, every 0.05 s for the **first second** after the pump starts, with
noise of 1 % - before the flow is steady.

**Three ways to find the viscosity.**

1. **Darcy**, the classical engineering way: take the last reading as the
   steady mean speed and solve $\bar u = GR^2/8\mu$ for $\mu$. It needs no
   model of the start-up, and it is only right if the flow has settled.
2. **Least squares on the exact solution**: adjust $C$ until Szymański's
   mean speed matches the readings. It is right, and it is possible only
   because this straight tube has an exact solution.
3. **The physics-informed network**: a fresh network for the velocity, and
   one more trainable number, $C = e^{\beta}$, an exponential so that it
   cannot go negative. It is **not an output of the network**: it is a
   number beside the weights, in the residual, changed by the optimiser with
   them. The loss is the residual plus the misfit between the network's mean
   speed, $\int_0^1 \hat u\,ds$, and the readings. $C$ starts at the nominal
   1.04, 29 % low.

In [ ]:
from scipy.optimize import least_squares
MU_TRUE = 3.5e-3                                              # Pa s: the coolant actually in the loop - unknown to the methods
c_true = MU_TRUE / pp.RHO * pp.T_END / pp.R_TUBE ** 2
T_REC = 0.25                                                  # the record: the first second (a quarter of the window)
t_m, q_m = pp.flow_meter(c_true, t_end_record=T_REC, every=T_REC / 20)   # 20 readings of the mean speed, scaled, with noise
to_mu = lambda c: c * pp.RHO * pp.R_TUBE ** 2 / pp.T_END      # scaled viscosity -> Pa s

mu_darcy = pp.G * pp.R_TUBE ** 2 / (8 * q_m[-1] * pp.V)       # 1. Darcy: the last reading taken as steady
fit = least_squares(lambda p: pp.exact_mean(t_m, c=p[0]) - q_m, [pp.C_SCALED])   # 2. the exact series fitted
mu_lsq = to_mu(fit.x[0])

gl_x, gl_w = np.polynomial.legendre.leggauss(16)              # 16 Gauss points across the section, for the mean speed
SQ = to_tensor(np.c_[np.tile(0.5 * (gl_x + 1), len(t_m)), np.repeat(t_m, 16)])   # (s, t) at every reading
WQ = to_tensor(0.5 * gl_w).reshape(-1)                        # their weights
QM = to_tensor(q_m).reshape(-1)                               # the readings
pts_rec = to_tensor(np.c_[rng.uniform(0, 1, 3000), T_REC * rng.uniform(0, 1, 3000) ** 2], requires_grad=True)   # points over the record only

class Inverse(torch.nn.Module):                               # the tube's network and one more trainable number
    def __init__(self):
        super().__init__()
        self.net = MLP(n_in=2, n_hidden=32, n_layers=4)       # a fresh dense tanh network: s and t in
        self.beta = torch.nn.Parameter(torch.tensor(float(np.log(pp.C_SCALED))))   # C = exp(beta), starts at the nominal
    def forward(self, p):
        s, t = p[:, 0:1], p[:, 1:2]
        return (1 - s) * t * self.net(torch.cat([s, t / T_REC], 1))   # the same trial function, time scaled to the record
inv = Inverse().to(DEVICE)                                    # the network and beta, on the device
W_D = 100.0                                                   # the weight of the readings
def inverse_loss():
    c = torch.exp(inv.beta)                                   # the viscosity being identified, always positive
    mean = (inv(SQ).reshape(len(t_m), 16) * WQ).sum(dim=1)    # the network's mean speed at each reading
    return residual(inv, pts_rec, c, pp.A_SCALED).pow(2).mean() + W_D * (mean - QM).pow(2).mean()
t0 = time.perf_counter()
# Adam first, then L-BFGS: the Part 2 training recipe
train_two_stage(inv, inverse_loss, adam_steps=500, lbfgs_steps=150, lr=2e-3, report_every=0)
inv_time = time.perf_counter() - t0
mu_pinn = to_mu(float(torch.exp(inv.beta)))
for name, mu in (("Darcy, last reading as steady", mu_darcy), ("least squares, exact series", mu_lsq), ("physics-informed network", mu_pinn)):
    print(f"  {name:32s}{1e3 * mu:6.2f} mPa s   ({100 * (mu / MU_TRUE - 1):+.1f} %)")
print(f"  true {1e3 * MU_TRUE:.2f} mPa s; the network trained in {inv_time:.0f} s")

**What you should see.** Darcy returns about 4.0 mPa·s, **15 % high**: after
one second the flow is not yet steady, the last reading is low, and a low
speed under a known pressure reads as a viscous liquid. The least-squares fit
and the network both return about 3.55 mPa·s, within 2 % - the error is the
noise of the readings, not the method.

---

## 7 · What the notebook says

* **Darcy's 64/Re is the steady limit of Newton's law in a tube.** The
  parabola gives it in two lines; it holds once the flow has settled and the
  entrance length after each bend has passed.
* **The start-up is exact, so the grid and the network fought on the truth.**
  Both met the agreed 1e-03; the grid did it with 21 unknowns in a
  millisecond and no training, the network a little more exactly after half
  a minute of training.
* **Three conditions were built into the network**: the wall and the start by
  factors, the symmetry about the axis by the input $s$. The loss was the
  equation alone.
* **The viscosity is the number worth finding.** It changes with temperature
  and with the mix, and it sets the pressure drop and the pump's work.
* **A classical formula used outside its conditions was the worst answer**:
  Darcy on a flow that had not settled was 15 % off. The exact series fitted
  by least squares was as good as the network. The network's way is the one
  that carries over to a bend or a milled channel, where no series exists -
  which is what the mini projects ask.
* **None of the numbers is a measured coolant.** The properties are typical
  values from memory; check them against the supplier's data before
  designing with them.

---

## 8 · Your report

**What these cells do.** You answer the questions below; the next cell writes `Ex09.1_report.md` with the results of section 5 and your answers, and the cell after it turns the report into a PDF for Moodle.

Each question builds part of an answer to one of L9.1's questions for the oral examination; the arrow says which. Keep each answer to a few sentences, with your numbers. Write your name in `NAME`, and each answer between the triple quotes that follow its question; an answer needs at least eight words.

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "Which terms of Newton's law for a liquid are left in a straight tube far from the bends, and why is the convective acceleration zero there? (-> L9.1 Q1)": """
""",
    "What is the Reynolds number of this flow, what does it tell you about laminar flow, and how long after a bend before the parabola - and Darcy's 64/Re - apply? (-> L9.1 Q2)": """
""",
    "Which three conditions did the trial function and the input s build into the network, and why was the residual divided by C? (-> L9.1 Q3)": """
""",
    "Why was Darcy's viscosity 15 % off while the least-squares fit and the network were within 2 %, and how would you have known, from the record alone, that Darcy could not be used? (-> L9.1 Q4)": """
""",
    "To conclude: for this straight tube the grid won on time and matched on accuracy. Name the situation in this cold plate where you would use the network instead, and say what you would measure to check it. (-> L9.1, Exercise slide)": """
""",
}

In [ ]:
MIN_WORDS = 8                                               # an answer is a few sentences, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:
    rows = ["| method | unknowns | worst error | online |", "|---|---|---|---|",
            f"| finite differences, {N_FDM} x {STEPS} | {N_FDM} | {e_fdm:.2e} | {1e3 * t_fdm:.2f} ms |",
            f"| PINN, 4 x 32 | {parameter_count(model)} | {e_pin:.2e} | {1e3 * t_pin:.2f} ms (trained offline in {train_time:.0f} s) |",
            "", "| viscosity from the first second | mPa s | off by |", "|---|---|---|",
            f"| Darcy, last reading as steady | {1e3 * mu_darcy:.2f} | {100 * (mu_darcy / MU_TRUE - 1):+.1f} % |",
            f"| least squares, exact series | {1e3 * mu_lsq:.2f} | {100 * (mu_lsq / MU_TRUE - 1):+.1f} % |",
            f"| physics-informed network | {1e3 * mu_pinn:.2f} | {100 * (mu_pinn / MU_TRUE - 1):+.1f} % |",
            "", f"True viscosity {1e3 * MU_TRUE:.2f} mPa s. The coolant's properties are typical values, not measurements."]
    out = ["# Ex_09.1 — The Pump Starts", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("Ex09.1_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figure of section 5, and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex09.1_report.md into Ex09.1_report.pdf, with any figure
# saved as Ex09.1_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "Ex09.1_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "Ex09.1_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("Ex09.1_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "Ex09.1_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass

---

## 9 · Mini project proposal

Two problems grown out of this set, for you to take further on your own. Each
student chooses **one** mini project from the Part 2 sets and has **one month**
to solve it individually. The dates and the hand-in are given in L13.

The ground truth is **given to you**, built by the course with a classical
solver and shown to have converged; the worked example below each project is
its first case. You do not solve the physics a second way: the month goes into
the deep learning - making a network predict what the ground truth says, as
accurately and as fast as the requirements ask.

### MP9.1A · One bend of the serpentine

**The problem.** The notebook's leg was straight. Between two legs the coolant turns through a
U-bend of 12 mm radius, and in a bend the fast liquid in the middle of the tube
is thrown outwards and comes back along the walls: two counter-rotating vortices
(Dean vortices) appear beside the axial flow, and the pressure drop rises above
Darcy's 64/Re. By how much, for this coolant at 0.3 L/min?

| | |
|---|---|
| tube, coolant, flow | as in the notebook: 6 mm, water–glycol at 40 °C, 0.3 L/min, Re = 452 |
| bend | radius 12 mm to the tube's axis: $a/R_c$ = 0.25, Dean number $\mathrm{Re}\sqrt{a/R_c}$ = 226 |
| asked | the axial velocity and the secondary flow across the section, fully developed; the friction factor in the bend |

The bend radius is a typical value, sourced before the data is released.

**Why a plain PINN is not enough.** The flow is three-dimensional: an axial velocity and a secondary flow across the
section, coupled - the secondary flow moves the axial velocity, and the axial
velocity drives the secondary flow through the centrifugal force. Two outputs,
fourth derivatives through a stream function or a pressure, and walls where both
the velocity and its slope vanish.

**What you build.** A PINN for the fully developed flow in the bend's cross-section: the axial velocity
and the secondary stream function as two outputs, in Dean's form of the equations,
with the no-slip wall built in. You find the friction factor and draw the two
vortices.

**The ground truth you get.** `MP9.1A_new_truth.npz`, from `tools/miniprojects/pipe_truth.py`: the same equations
solved by finite volumes on a polar grid with Newton's method (`dean.py`), at
48 × 96 cells. Checked against Dean's own series at small Dean numbers (0.99623
against 0.99649 at $C$ = 100), and by refining the grid.

**Worked example.** 

| cells | friction factor over 64/Re | pressure drop |
|---|---|---|
| 24 × 48 | 2.014 | 791 Pa/m |
| 32 × 64 | 2.007 | 789 Pa/m |
| 48 × 96 | 1.995 | 784 Pa/m |

In the bend the pressure drop is **twice** the straight tube's 393 Pa/m. Mori and
Nakayama's correlation, quoted from memory, gives 2.02 at this Dean number.

**Requirements.**

* The friction factor in the bend within **3 %** of the ground truth's.
* The axial velocity within **2 %** of the straight tube's centre speed everywhere in the section.
* The two vortices drawn, and the position of the fastest axial velocity: towards the outer wall.

**What you hand in.** The axial velocity and the secondary flow against the ground truth, the friction factor, and a paragraph on what the bend costs the pump compared with a straight leg.

### MP9.1B · The milled channel

**The problem.** Many cold plates have no tube: the channel is milled into the aluminium and
closed by a lid, so its section is a rectangle, 4 × 2 mm here. The axial
velocity depends on two coordinates across it, and the corners hold the liquid
back. When the pump starts, how does the velocity grow across the rectangle,
and what is the friction factor once it is steady?

| | |
|---|---|
| channel | 4 mm wide, 2 mm high, milled; hydraulic diameter 2.67 mm |
| coolant, flow | as in the notebook: water–glycol at 40 °C, 0.3 L/min, a mean speed of 0.63 m/s, Re = 710 |
| start | at rest; the pressure drop that carries 0.3 L/min applied at $t$ = 0 and held |

The channel's size is a typical value, sourced before the data is released.

**Why a plain PINN is not enough.** Two coordinates across the section and one in time; four walls that meet in
corners, where the velocity is held at zero from two sides; and no symmetry
variable that builds the condition in, as $s$ did for the tube.

**What you build.** A PINN for $u(x, y, t)$ with the four walls and the start built in by factors, as
the notebook did for the tube, trained over the start-up. You find the steady
friction factor and the time to 90 % of the steady flow.

**The ground truth you get.** `MP9.1B_new_truth.npz`, from `tools/miniprojects/pipe_truth.py`: the exact solution,
a double sine series in $x$ and $y$ with an exponential in time for each term,
81 × 81 terms, on 161 × 81 points at 23 instants.

**Worked example.** The pressure drop that carries 0.3 L/min is **6834 Pa/m** - seventeen times the
round tube's, because the channel's section is smaller. $f\,\mathrm{Re}$ = **62.21**
on the hydraulic diameter, against 64 for a round tube; the centre speed is 1.25 m/s.
Started from rest, the flow is half-way after 0.09 s, at 90 % after 0.31 s and at
99 % after 0.63 s - six times faster than the 6 mm tube, because the channel is smaller.

**Requirements.**

* The velocity within **1 %** of the steady centre speed everywhere, at every recorded instant.
* $f\,\mathrm{Re}$ within **1 %** of 62.21.
* The time to 90 % of the steady flow within **0.02 s**.

**What you hand in.** The velocity across the channel against the exact solution at three instants, the friction factor, the times to 50 % and 90 %, and a paragraph comparing the channel with the round tube.